In [ ]:
!pip -q install transformers accelerate matplotlib

In [ ]:
import math,torch,numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer,AutoModelForCausalLM

In [ ]:
Model_id="Qwen/Qwen2.5-0.5B-Instruct"
device="cuda" if torch.cuda.is_available() else "cpu"
dtype=torch.float16 if torch.cuda.is_available() else torch.float32
tokenizer=AutoTokenizer.from_pretrained(Model_id,trust_remote_code=True)
model=AutoModelForCausalLM.from_pretrained(Model_id,trust_remote_code=True).to(device).to(dtype)
N_LAYERS=len(model.model.layers)
Layer=N_LAYERS//2

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [ ]:
class EndocrineSystem:
  Names=["dopamine","cortisol","serotonin","oxytocin"]

  def __init__(self):
    self.tau={"dopamine":5,"cortisol":11,"serotonin":15,"oxytocin":2} #ne kadar hızlı tepkı verecegını ve ne kadar surecegını belırlıyoruz
    self.g={"dopamine":0.2,"cortisol":0.1,"serotonin":0.3,"oxytocin":0.4} #toplamı bır olmak zorunda ağırlıklarını tanımlayıp onemını ve bırbırlerıne olan etkısını belırlıyoruz

  def reset (self):
    self.level=dict(self.base)
    self.history=[dict(self.level)]

  def step(self,stim:dict):
    prev=dict(self.level)
    new = {}
    for h in self.Names:
      b = self.base[h]
      x = b+(prev[h]-b)*math.exp(-1/self.tau[h])
      x+=stim.get(h,0)
      new[h]=x

    new["dopamine"]-=0.11*prev["cortisol"]
    new["cortisol"]-=0.15*prev["dopamine"]  #burada hormonlar arası etkılesımı gırıyoruz bırı dıgerını arttırırken dıgerı azaltma gıbı tanımlamalarını yapıyoruz.
    new["serotonin"]-=0.11*prev["oxytocin"] #prev etkılesıme sokmak demek

    self.level={h:float(np.clip(v,0,1)) for h,v in new.items()}
    self.history.append(dict(self.level))
    return self.level

  def deviation(self): #kullanmadıgımız zamanlarda yavas yavas normale donmesı için kullanırız.
    return {h:abs(self.level[h]-self.base[h]) for h in self.Names}

endo=EndocrineSystem()


In [ ]:
LEXICON={
    "praise":(["great","amazing","awesome","brilliant","love","perfect","well done"],
              {"dopamine":0.55,"serotonin":0.03,"oxytocin":0.15}),
    "insult":(["stupid","useless","idiot","hate","terrible","worst","shut up"],
              {"cortisol":0.40,"serotonin":-0.06,"oxytocin":-0.10}),
    "thanks":(["thank","thanks","appreciate","grateful","kind","friend"],
              {"oxytocin":0.45,"serotonin":0.08,"dopamine":0.10}),
}

def message_to_stimulus(msg:str)-> dict:
  m=msg.lower()
  stim={}
  for _, (words,effect) in LEXICON.items():
    if any(w in m for w in words):
      stim.update(effect)
  return stim

In [ ]:
Contrast={
    "dopamine":[
        ("I'm so excited, this is fantastic and I can't wait!","This is fine, nothing special, whatever,"),
        ("Wow, what a thrilling idea, let's go!","Okay.I suppose that could work."),
        ("I feel energized and eager to start.","I feel flat and unmotivated"),
    ],
    "cortisol":[
        ("I'm tense and on guard, something feels threatening.","I'm relaxed and feel completely safe"),
        ("Back off , I don't trust you at all.","Sure, I trust you, no worries."),
        ("I'm stessed, irritated and defensive.","I'm relaxed and calm"),
    ],
    "serotonin":[
        ("I feel content, stable and at peace", "I feel restless,anxious and unsettled"),
        ("Everything is balanced and okay.","Everything feels off chaotic"),
        ("I'm patient and steady today","I'm jittery and impatient today"),
        ],
    "oxytocin":[
        ("I really care about you, my dear friend.","You are just another user to me"),
        ("I feel warm and close to you","I feel distant and indifferent toward you"),
        ("We're in this together, I've got you","Handle it yourself, not my problem")
    ],
}

@torch.no_grad()
def mean_activation(text):
  # Giriş metnini tokenize edip aktif cihaza gönderiyoruz
  ids = tokenizer(text, return_tensors="pt").to(device)
  # Doğru parametre adı: output_hidden_states
  out = model(**ids, output_hidden_states=True)
  return out.hidden_states[Layer+1][0].float().mean(dim=0)

vectors = {}
for h, pairs in Contrast.items():
  # p: pozitif (yüksek), n: negatif (düşük/zıt) durum
  diffs = [mean_activation(p) - mean_activation(n) for p, n in pairs]
  V = torch.stack(diffs).mean(0)
  vectors[h] = V / V.norm()

Resid_norm = mean_activation("Hello , how are you today?").norm().item()

names = list(vectors.keys())
# PyTorch kütüphane adı düzeltildi ve kosinüs benzerliği matrisi matris çarpımı ile hesaplandı
sim = torch.stack([vectors[a] for a in names]) @ torch.stack([vectors[b] for b in names]).T
print("residual norm",round(Resid_norm,3))

residual norm 241.792


In [ ]:
strentgh=0.6

def steering_hook(module,inputs,output):
  dev=endo.deviation()
  delta=sum(dev[h]*Vectors[h] for h in Vectors)*Strength*Resid_Norm
  if isinstance(output,tuple):
    return(output[0]+delta.to(output[0].dtype),output[1:])
  return output+delta.to(output.dtype)

_handle=None
def steering(on:bool):
  global _handle
  if on and _handle is None:
    _handle=model.model.layers[Layer].register_forward_hook(steering_hook)
  if not on and _handle is not None:
    _handle.remove(); _handle=None

In [ ]:
system="you are a helpful asistant. Reply in 1-2 sentences"